In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Dane wejściowe i wyjściowe Estimator

<Accordion>
<AccordionItem title="Wersje pakietów">

Kod na tej stronie został opracowany przy użyciu następujących wymagań.
Zalecamy używanie tych lub nowszych wersji.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Ta strona zawiera przegląd danych wejściowych i wyjściowych prymitywu Qiskit Runtime Estimator, który wykonuje obciążenia robocze na zasobach obliczeniowych IBM Quantum&reg;. Estimator pozwala efektywnie definiować zwektoryzowane obciążenia robocze za pomocą struktury danych zwanej [**Primitive Unified Bloc (PUB)**](/guides/primitive-input-output#pubs). Są one używane jako dane wejściowe do metody [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) prymitywu Estimator, który wykonuje zdefiniowane obciążenie robocze jako zadanie. Następnie, po zakończeniu zadania, wyniki są zwracane w formacie zależnym zarówno od użytych PUB-ów, jak i opcji środowiska uruchomieniowego określonych przez prymityw.

## Dane wejściowe
Każdy PUB ma ten format:

(`<pojedynczy obwód>`, `<jeden lub więcej obserwabli>`, `<opcjonalne wartości parametrów>`, `<opcjonalna precyzja>`),

Opcjonalne `wartości parametrów` mogą być listą lub pojedynczym parametrem. Elementy z obserwabli i wartości parametrów są łączone zgodnie z regułami rozgłaszania NumPy opisanymi w temacie [Dane wejściowe i wyjściowe prymitywu](primitive-input-output#broadcasting-rules), i zwracana jest jedna szacowana wartość oczekiwana dla każdego elementu rozgłoszonego kształtu.

> **Note:** Jeśli dane wejściowe zawierają pomiary, są one ignorowane.

Dla prymitywu Estimator PUB może zawierać co najwyżej cztery wartości:
- Pojedynczy `QuantumCircuit`, który może zawierać jeden lub więcej obiektów [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- Listę jednego lub więcej obserwabli, które określają szacowane wartości oczekiwane, ułożone w tablicę (na przykład pojedyncza obserwabla reprezentowana jako tablica 0-wymiarowa, lista obserwabli jako tablica 1-wymiarowa itd.). Dane mogą być w dowolnym formacie `ObservablesArrayLike`, takim jak `Pauli`, `SparsePauliOp`, `PauliList` lub `str`.
   > **Note:** - Przemienające obserwable **w tym samym PUB** są grupowane razem za pomocą [tej metody](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting).
>    - Przemienające obserwable w różnych PUB-ach, nawet jeśli mają ten sam obwód, nie są szacowane za pomocą tego samego pomiaru. Każdy PUB reprezentuje inną bazę pomiaru, dlatego dla każdego PUB wymagane są osobne pomiary.
>    - Aby upewnić się, że przemienające obserwable są szacowane za pomocą tego samego pomiaru, zgrupuj je w tym samym PUB.
- Kolekcja wartości parametrów do powiązania z obwodem. Można ją określić jako pojedynczy obiekt podobny do tablicy, gdzie ostatni indeks jest nad obiektami `Parameter` obwodu, lub pominąć (lub równoważnie ustawić na `None`), jeśli obwód nie ma obiektów `Parameter`.
- (Opcjonalnie) Docelowa precyzja szacowania wartości oczekiwanych
---

Poniższy kod demonstruje przykładowy zestaw zwektoryzowanych danych wejściowych dla prymitywu `Estimator` i wykonuje je na backendzie IBM&reg; jako pojedynczy obiekt `RuntimeJobV2`.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## Dane wyjściowe
Po wysłaniu jednego lub więcej PUB-ów do QPU i pomyślnym zakończeniu zadania dane są zwracane jako obiekt kontenera [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) dostępny przez wywołanie metody `RuntimeJobV2.result()`.

`PrimitiveResult` zawiera iterowalną listę obiektów [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult), które zawierają wyniki wykonania dla każdego PUB.

Każdy element tej listy odpowiada każdemu PUB przesłanemu do metody `run()` prymitywu (na przykład zadanie przesłane z 20 PUB-ami zwróci obiekt `PrimitiveResult` zawierający listę 20 obiektów `PubResult`, po jednym odpowiadającym każdemu PUB).

Każdy `PubResult` dla prymitywu Estimator zawiera co najmniej tablicę wartości oczekiwanych (`PubResult.data.evs`) i powiązanych odchyleń standardowych (albo `PubResult.data.stds`, albo `PubResult.data.ensemble_standard_error` w zależności od użytego `resilience_level`), ale może zawierać więcej danych w zależności od opcji łagodzenia błędów, które zostały określone.

Każdy obiekt `PubResult` posiada zarówno atrybut `data`, jak i `metadata`.
    - Atrybut `data` to dostosowany [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin), który zawiera rzeczywiste wartości pomiaru, odchylenia standardowe i tym podobne.
    - `DataBin` ma różne atrybuty w zależności od kształtu lub struktury powiązanego PUB oraz opcji łagodzenia błędów określonych przez prymityw użyty do przesłania zadania (na przykład [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) lub [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)).
    - Atrybut `metadata` zawiera informacje o opcjach środowiska uruchomieniowego i łagodzenia błędów (wyjaśnione później w sekcji [Metadane wyników](#result-metadata) na tej stronie).

Poniżej przedstawiono wizualny zarys struktury danych `PrimitiveResult` dla danych wyjściowych Estimator:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

Mówiąc prosto, pojedyncze zadanie zwraca obiekt `PrimitiveResult` i zawiera listę jednego lub więcej obiektów `PubResult`. Te obiekty `PubResult` przechowują następnie dane pomiaru dla każdego PUB przesłanego do zadania.

Poniższy fragment kodu opisuje format `PrimitiveResult` (i powiązanego `PubResult`) dla zadania utworzonego powyżej.

In [2]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
